# BERT -- Masked Language Modeling

GPT predicts the next word. BERT perdicts a missing word. One sentence of difference -- and half a decade of everything embedding-shaped.

## Problem Definition

What it we took a transformer encoder, trained it on every sentence on the internet, and forced it to predict missing owrds from context on both sides? Then you fine-tune one head on your downstream task.

In 2026 encoder-only models are still the tight tool for classification, retrieval, and structured extration -- they run 5-10x faster per token than decoders.

## Basic Concept

Masked language modeling: Pick tokens, mask then, predict originals.

### Training Sample

```
input:  the [MASK] brown fox jumps [MASK] the lazy dog.
target: the quick brown fox jumps over the lazy dog.
```
Train the model to predict the original tokens at masked position. Because the encoder is bidirectional, predicting [MASK] at position 1 can use `brown fox jumps` at position 2+.

### BERT mask rules

Of the 15% of tokens selected for prediction:
* 80% are replaced with [MASK]
* 10% are replaced with a random token
* 10% are left unchanged.

Why not always [MASK]? Because [MASK] never appears at inference time. Training the model to expect [MASK] at 100% of masked positions would create a distribution shift between pretraining and fine-tuning. The 10% random + 10% unchanged keeps the model honest.

### What changed in 2026: ModernBERT

The 2024 ModernBERT paper rebuilt the block with 2026 primitives:

| Component | Original BERT (2018) | ModernBERT (2024) |
|-----------|----------------------|-------------------|
| Positional | Learned absolute | RoPE |
| Activation | GELU | GeGLU |
| Normalization | LayerNorm | Pre-norm RMSNorm |
| Attention | Full dense | Alternating local (128) + global |
| Context length | 512 | 8192 |
| Tokenizer | WordPiece | BPE |

And unlike the 2018 stack, it is Flash-Attention-native. Inference is 2–3× faster at sequence length 8K than DeBERTa-v3 with better GLUE scores.

### Use cases that still pick an encoder in 2026

| Task | Why encoder beats decoder |
|------|---------------------------|
| Retrieval / semantic search embeddings | Bidirectional context = better embedding quality per token |
| Classification (sentiment, intent, toxicity) | One forward pass; no generation overhead |
| NER / token labeling | Per-position output, natively bidirectional |
| Zero-shot entailment (NLI) | Classifier head on top of encoder |
| Reranker for RAG | Cross-encoder scoring, 10x faster than LLM rerankers |


# Build your Own

In [ ]:
import random
from collections import Counter

MASK_ID = 0
CLS_ID = 1
SEP_ID =2
SPECIAL_IDS = {MASK_ID, CLS_ID, SEP_ID}
IGNORE_INDEX = -100

def create_mlm_batch(tokens, vocab_size, mask_prob=0.15, rng=None):
    if rng is None:
        rng = random.Random()
    input_ids = list(tokens)
    labels = [IGNORE_INDEX] * len(tokens)
    for i, t in enumerate(tokens):
        if t in SPECIAL_IDS:
            continue
        if rng.random() >= mask_prob:
            continue
        labels[i] = t
        r = rng.random()
        if r < 0.8:
            input_ids[i] = MASK_ID
        elif r < 0.9:
            rand_id = t
            while rand_id in SPECIAL_IDS or rand_id == t:
                rand_id = rng.randrange(vocab_size)
            input_ids[i] = rand_id
    return input_ids, labels

import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

with SectionPrinter("BERT Masking"):
    FAKE_VOCAB_SIZE = 20
    FAKE_SEQ_LEN = 10
    # Fake tokenize
    def tokenize():
        return [random.randint(5, FAKE_VOCAB_SIZE) for _ in range(FAKE_SEQ_LEN)]

    # Tokenize
    tokens = tokenize()
    print(tokens)

    # Create a batch
    input_ids, labels = create_mlm_batch(tokens, FAKE_VOCAB_SIZE)
    print(input_ids)
    print(labels)



========================BERT Masking========================
[14, 19, 19, 15, 15, 19, 16, 6, 9, 19]
[14, 19, 8, 15, 15, 0, 16, 6, 0, 19]
[-100, -100, 19, -100, -100, 19, -100, -100, 9, -100]


In [ ]:
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
model = AutoModel.from_pretrained("answerdotai/ModernBERT-base")

text = "Attention is all you need."
inputs = tok(text, return_tensors="pt")
out = model(**inputs).last_hidden_state

print(out.shape)

# Do your own work with hidden state...

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

W0808 16:12:10.434000 29211 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0808 16:12:10.482000 29211 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0808 16:12:10.506000 29211 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 9, 768])
